# 10年定着予測 - CPUでのフルアブレーション再現 + アンサンブル比重のパラメータチューニング

**背景**: `20_eda_v3_driven_features`はGPUで baseline/E_memo/F_grade_consistency/G_self_study/combo_EFG
の5構成を検証し、combo_EFGがPublic 0.537215で総合最良を更新した。しかし`21_engagement_deepdive_features`で
判明した通り、**GPU/CPU・異なるCPUアーキテクチャ間で検証スコアに無視できない差（約0.008）が生じる**ため、
GPUの数値とCPUの数値を混在させて比較するのは methodologically 問題がある。

本ノートブックでは以下2点を行う。

1. **GPU時と同じ5構成（baseline / E_memo / F_grade_consistency / G_self_study / combo_EFG）を、
   今回はCPUで一貫して再学習し直す**（GPU/CPU混在比較を避け、同一環境内で完結させる）。
   実行時間短縮のため学習構成の数自体は`20_`から変えていないが、CPU実行により大幅に高速化される見込み
   （`21_`ではCPU4構成が約8〜10分で完了した実績あり）。
2. **アンサンブルの重みをパラメータチューニングする**。過去の教訓（`12_`でStacking/Optimized Weighted
   AverageがOOFでは良く見えてもPublicでは単純平均や単体モデルに劣った）を踏まえ、
   `12_`と同じ`scipy.optimize`によるOOF Log Loss最小化に加え、
   **80/20・75/25の2つのsplitそれぞれで独立に重みを最適化し、重みベクトル自体が両splitで
   一貫しているかを確認する**（一貫していなければ過学習の疑いが強いと判断する）。

## 提出優先度
E_memo単体 > G_self_study単体の順で提出する（20_の検証結果でE_memoの方が改善幅が大きいため）。

## 実行環境
Google Colab（CPU、ハイメモリ推奨）を想定。

In [1]:
!pip install -q catboost optuna

In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [4]:
import datetime
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from scipy.optimize import minimize
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
SCRIPT_NAME = "22_ensemble_weight_tuning"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-10 10:20:29] [INFO] === [22_ensemble_weight_tuning] 実験開始 ===


INFO:22_ensemble_weight_tuning:=== [22_ensemble_weight_tuning] 実験開始 ===


[2026-08-10 10:20:30] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260810


INFO:22_ensemble_weight_tuning:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260810


[2026-08-10 10:20:30] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/22_ensemble_weight_tuning_checkpoint.csv


INFO:22_ensemble_weight_tuning:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/22_ensemble_weight_tuning_checkpoint.csv


[2026-08-10 10:20:30] [INFO] チェックポイントは未作成（新規実行）


INFO:22_ensemble_weight_tuning:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-10 10:20:34] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:22_ensemble_weight_tuning:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-10 10:20:34] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:22_ensemble_weight_tuning:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-10 10:20:34] [INFO] 定着率: 0.5647


INFO:22_ensemble_weight_tuning:定着率: 0.5647


[2026-08-10 10:20:34] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:22_ensemble_weight_tuning:Train IDs: 2761, Test IDs: 2502


## 1. 基本特徴量関数の定義（split非依存、`18_`と同一ロジック）

In [7]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [8]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-10 10:20:35] [INFO] ------------------------------------------------------------


INFO:22_ensemble_weight_tuning:------------------------------------------------------------


[2026-08-10 10:20:35] [INFO] split非依存の基本特徴量を生成中...


INFO:22_ensemble_weight_tuning:split非依存の基本特徴量を生成中...


[2026-08-10 10:20:35] [INFO] ------------------------------------------------------------


INFO:22_ensemble_weight_tuning:------------------------------------------------------------


[2026-08-10 10:28:31] [INFO] split非依存の基本特徴量生成完了


INFO:22_ensemble_weight_tuning:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [9]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    """文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）"""
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-10 10:28:32] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:22_ensemble_weight_tuning:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-10 10:28:34] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:22_ensemble_weight_tuning:入社時メモ: SVD累積寄与率=0.760


[2026-08-10 10:28:39] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:22_ensemble_weight_tuning:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-10 10:28:42] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:22_ensemble_weight_tuning:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-10 10:28:42] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:22_ensemble_weight_tuning:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [10]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-10 10:28:42] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:22_ensemble_weight_tuning:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-10 10:31:42] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:22_ensemble_weight_tuning:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [11]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-10 10:31:42] [INFO] Persona単位の基本特徴量を生成中...


INFO:22_ensemble_weight_tuning:Persona単位の基本特徴量を生成中...


[2026-08-10 10:31:42] [INFO] Persona単位の基本特徴量処理完了


INFO:22_ensemble_weight_tuning:Persona単位の基本特徴量処理完了


## 5. 自己学習（詳細）特徴量（ブロックG、EDA v3 分析7由来、split非依存）

`自己学習（詳細）`列（例:「クラウド基盤入門：2.0時間｜統計学基礎：1.5時間」）はこれまで一度も
特徴量化されていなかった列。「｜」区切りで複数テーマが並記される行を正しく分割してから
テーマ・時間を抽出する（単純な単一マッチだと2つ目以降のテーマ・時間を取りこぼすバグがあるため注意）。
テーマと初期職種のマッチ度はEDA v3で非有意だったため、集計特徴量（実施月数・合計時間・ユニークテーマ数）
のみを採用する。

In [12]:
def parse_all_study_themes(s):
    if pd.isna(s) or s == "受講なし":
        return [], 0.0
    parts = str(s).split("｜")
    themes, total = [], 0.0
    for part in parts:
        m = re.match(r"(.+?)：([\d.]+)時間", part)
        if m:
            themes.append(m.group(1))
            total += float(m.group(2))
    return themes, total


def create_self_study_features(monthly_df, employee_ids):
    """自己学習実施月数・合計時間・ユニークテーマ数を集計（EDA v3 分析7と同一ロジック）"""
    df = monthly_df[["社員ID", "自己学習（詳細）"]].copy()
    parsed = df["自己学習（詳細）"].apply(parse_all_study_themes)
    df["_themes"] = parsed.apply(lambda x: x[0])
    df["_hours"] = parsed.apply(lambda x: x[1])

    total_hours = df.groupby("社員ID")["_hours"].sum()
    active_months = df[df["_hours"] > 0].groupby("社員ID").size()
    unique_themes = df.groupby("社員ID")["_themes"].apply(lambda s: len(set(t for tl in s for t in tl)))

    out = pd.DataFrame({
        "自己学習合計時間": total_hours,
        "自己学習実施月数": active_months,
        "自己学習ユニークテーマ数": unique_themes,
    })
    out = out.reindex(employee_ids).fillna(0.0).reset_index().rename(columns={"index": "社員ID"})
    return out

logger.info("自己学習（詳細）特徴量(ブロックG)を生成中...")
train_selfstudy = create_self_study_features(train_monthly, train_ids)
test_selfstudy = create_self_study_features(test_monthly, test_ids)
logger.info(f"ブロックG: Train {train_selfstudy.shape}, Test {test_selfstudy.shape}")
train_selfstudy.head()

[2026-08-10 10:31:42] [INFO] 自己学習（詳細）特徴量(ブロックG)を生成中...


INFO:22_ensemble_weight_tuning:自己学習（詳細）特徴量(ブロックG)を生成中...


[2026-08-10 10:31:43] [INFO] ブロックG: Train (2761, 4), Test (2502, 4)


INFO:22_ensemble_weight_tuning:ブロックG: Train (2761, 4), Test (2502, 4)


,社員ID,自己学習合計時間,自己学習実施月数,自己学習ユニークテーマ数
0,E000001,48.5,9.0,7
1,E000005,10.5,3.0,4
2,E000007,61.0,13.0,11
3,E000008,27.0,8.0,6
4,E000010,39.0,11.0,15


## 6. 入社時メモ構造化特徴量（ブロックE、EDA v3 分析2由来、split非依存）

`入社時メモ`の「キャリア志向」「勤務地・働き方」セクションを正規表現でパースし、
キャリア志向カテゴリ・転居許容・在宅希望・希望勤務地をカテゴリ変数として抽出する
（TF-IDF/SVDという間接表現に加え、直接的なカテゴリ変数として補強する狙い）。
抽出できなかった場合は"unknown"として扱う（CatBoostのカテゴリ変数としてそのまま利用できる）。

In [13]:
def extract_memo_section(text, section_name):
    if pd.isna(text):
        return None
    m = re.search(rf"{section_name}：(.+?)(?:\n|$)", text)
    return m.group(1).strip() if m else None


def classify_career_orientation(s):
    if s is None:
        return "unknown"
    if "限定していない" in s or "限定しない" in s:
        return "未定"
    if "管理職" in s:
        return "管理職志向"
    if "専門職" in s:
        return "専門職志向"
    if "安定" in s:
        return "安定志向"
    return "other"


def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


_NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
_POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_relocation(s):
    if s is None:
        return "unknown"
    if _NEG_RELOC.search(s):
        return "false"
    if _POS_RELOC.search(s):
        return "true"
    return "unknown"


_NEG_REMOTE = re.compile(r"在宅勤務を(必須条件としていない|希望しない|希望していない|希望せず)")
_POS_REMOTE = re.compile(r"在宅勤務を希望")


def classify_remote_pref(s):
    if s is None:
        return "unknown"
    if _POS_REMOTE.search(s):
        return "true"
    if _NEG_REMOTE.search(s):
        return "false"
    return "unknown"


def extract_desired_location(s):
    if s is None:
        return "unknown"
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else "unknown"


def create_memo_structured_features(persona_df):
    career_section = persona_df["入社時メモ"].apply(lambda t: extract_memo_section(t, "キャリア志向"))
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "memo_career_cat": career_section.apply(classify_career_orientation).values,
        "memo_転居許容": ws_section.apply(classify_relocation).values,
        "memo_在宅希望": ws_section.apply(classify_remote_pref).values,
        "memo_希望勤務地": ws_section.apply(extract_desired_location).values,
    })

logger.info("入社時メモ構造化特徴量(ブロックE)を生成中...")
train_memo = create_memo_structured_features(train_persona)
test_memo = create_memo_structured_features(test_persona)
logger.info(f"ブロックE: Train {train_memo.shape}, Test {test_memo.shape}")
train_memo.head()

[2026-08-10 10:31:43] [INFO] 入社時メモ構造化特徴量(ブロックE)を生成中...


INFO:22_ensemble_weight_tuning:入社時メモ構造化特徴量(ブロックE)を生成中...


[2026-08-10 10:31:43] [INFO] ブロックE: Train (2761, 5), Test (2502, 5)


INFO:22_ensemble_weight_tuning:ブロックE: Train (2761, 5), Test (2502, 5)


,社員ID,memo_career_cat,memo_転居許容,memo_在宅希望,memo_希望勤務地
0,E000001,未定,true,true,北海道
1,E000005,未定,false,false,東京
2,E000007,未定,false,false,東京
3,E000008,管理職志向,true,true,東京
4,E000010,安定志向,false,false,東京


## 7. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"E"}`/`{"F"}`/`{"G"}`/`{"E","F","G"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の最良構成）に対してブロックを単体・組み合わせで追加できるようにする。

ブロックF（経験等級整合性）は目的変数を直接使わないため部署Target Encodingのようなリークの危険はないが、
念のため他の派生特徴量（職種別偏差・等級内偏差等）と同様に**回帰は学習期間のIDのみでfitし、
チューニング期間・Testへはそのモデルで予測を適用する**という統一ルールに従う。

In [14]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    """初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）"""
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None):
    """指定した分割比率で特徴量を組み立てる。extra_blocks: {"E","F","G"}のサブセット"""
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "E" in extra_blocks:
        tf = tf.merge(train_memo, on=ID_COL, how="left")
        ttf = ttf.merge(test_memo, on=ID_COL, how="left")

    if "G" in extra_blocks:
        tf = tf.merge(train_selfstudy, on=ID_COL, how="left")
        ttf = ttf.merge(test_selfstudy, on=ID_COL, how="left")

    if "F" in extra_blocks:
        mid_fit = tf[(tf[ID_COL].isin(train_period_ids)) & (tf["入社区分"] == "中途")]
        reg = LinearRegression().fit(mid_fit[["前職経験月数"]].values, mid_fit["初期等級_num"].values)
        for df_ in [tf, ttf]:
            is_mid = (df_["入社区分"] == "中途")
            df_["経験等級_残差"] = np.nan
            if is_mid.sum() > 0:
                df_.loc[is_mid, "経験等級_残差"] = (
                    df_.loc[is_mid, "初期等級_num"].values - reg.predict(df_.loc[is_mid, ["前職経験月数"]].values)
                )
            df_["is_中途"] = is_mid.astype(int)

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了")

✅ 部署Target Encoding・prepare_split関数定義完了


## 8. チェックポイント機能（`18_`と同一）

In [15]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=["config", "n_features", "val_score", "submission_path"])

def save_checkpoint_row(result):
    df = pd.DataFrame([result])
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']:.6f}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了")

✅ チェックポイント関数定義完了


## 9. モデル実行関数（CatBoost、CPU）

`18_`のステップBでCatBoostが大差で最良だったため、本ノートブックではCatBoostのみで検証する。
探索範囲（n_trials, パラメータレンジ）は`18_`/`20_`と完全に同一のまま、`task_type`のみ`GPU`→`CPU`に
変更する（`21_`で確認した通り、Train 2,761行という規模ではCPUの方が高速）。

In [16]:
def run_model_config(ag_train_data, ag_tuning_data, test_features, config_label, n_trials=25):
    feature_cols = [c for c in ag_train_data.columns if c not in ["入社日", TARGET_COL]]
    obj_cols = [c for c in feature_cols if ag_train_data[c].dtype == "object"]

    X_tr = ag_train_data[feature_cols].fillna(-999)
    y_tr = ag_train_data[TARGET_COL]
    X_va = ag_tuning_data[feature_cols].fillna(-999)
    y_va = ag_tuning_data[TARGET_COL]
    X_test = test_features[feature_cols].fillna(-999)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params

    final_model = cb.CatBoostClassifier(
        **best_params, iterations=3000, random_seed=SEED, verbose=False,
        cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
    )
    final_model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
    val_preds = final_model.predict_proba(X_va)[:, 1]
    test_preds = final_model.predict_proba(X_test)[:, 1]

    val_score = log_loss(y_va, val_preds)
    sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    sub = pd.DataFrame({ID_COL: test_features.index, TARGET_COL: test_preds})
    sub.to_csv(sub_path, index=False, header=False)

    val_pred_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_valpreds.npy"
    np.save(val_pred_path, val_preds)

    logger.info(f"[{config_label}] n_features={len(feature_cols)}, val_score={val_score:.6f}")
    return {
        "config": config_label, "n_features": len(feature_cols), "val_score": val_score,
        "submission_path": str(sub_path), "val_pred_path": str(val_pred_path),
    }

print("✅ run_model_config関数定義完了")

✅ run_model_config関数定義完了


## 10. アブレーション: baseline（`18_`最良構成） / E単体 / F単体 / G単体 / combo_EFG × 2 split（CPU再学習）

`20_`のGPU実行と同じ5構成を、今回はCPUで一貫して学習し直す。GPU実測値（`data/output/submit_result_report.md`
参照）とは単純比較せず、本ノートブック内の数値同士（CPU同士）でのみ比較する。

In [17]:
SPLIT_RATIOS = {"split_80_20": 0.8, "split_75_25": 0.75}
BLOCK_CONFIGS = {
    "baseline": set(),
    "E_memo": {"E"},
    "F_grade_consistency": {"F"},
    "G_self_study": {"G"},
    "combo_EFG": {"E", "F", "G"},
}

ablation_results = []
for split_name, ratio in SPLIT_RATIOS.items():
    for block_name, blocks in BLOCK_CONFIGS.items():
        config_label = f"{split_name}_{block_name}"
        def _run(ratio=ratio, blocks=blocks, config_label=config_label):
            logger.info(f"=== {config_label} ===")
            ag_train_data, ag_tuning_data, test_features_full = prepare_split(ratio, extra_blocks=blocks)
            return run_model_config(ag_train_data, ag_tuning_data, test_features_full, config_label, n_trials=25)
        result = run_or_resume(config_label, _run)
        result["split"] = split_name
        result["block_config"] = block_name
        ablation_results.append(result)

ablation_df = pd.DataFrame(ablation_results)
ablation_pivot = ablation_df.pivot(index="block_config", columns="split", values="val_score")
ablation_pivot["mean"] = ablation_pivot[["split_80_20", "split_75_25"]].mean(axis=1)
ablation_pivot["std"] = ablation_pivot[["split_80_20", "split_75_25"]].std(axis=1)
ablation_pivot = ablation_pivot.reindex(["baseline", "E_memo", "F_grade_consistency", "G_self_study", "combo_EFG"])
ablation_pivot["mean_diff_vs_baseline"] = ablation_pivot["mean"] - ablation_pivot.loc["baseline", "mean"]
ablation_pivot = ablation_pivot.sort_values("mean")

logger.info("=" * 60)
logger.info("アブレーション結果（baseline vs E/F/G単体 vs combo_EFG）")
logger.info("=" * 60)
logger.info("\n" + ablation_pivot.to_string())
print("\n■ アブレーション結果:")
print(ablation_pivot.to_string())
print("\n※ このアブレーションはCPU同士の比較なので、GPU実測値(data/output/submit_result_report.md参照)とは直接比較しない")

[2026-08-10 10:31:44] [INFO] === split_80_20_baseline ===


INFO:22_ensemble_weight_tuning:=== split_80_20_baseline ===


[2026-08-10 10:36:26] [INFO] [split_80_20_baseline] n_features=439, val_score=0.554220


INFO:22_ensemble_weight_tuning:[split_80_20_baseline] n_features=439, val_score=0.554220


[2026-08-10 10:36:26] [INFO] === split_80_20_E_memo ===


...

[2026-08-10 11:05:12] [INFO] ============================================================


INFO:22_ensemble_weight_tuning:============================================================


[2026-08-10 11:05:12] [INFO] 
split                split_75_25  split_80_20      mean       std  mean_diff_vs_baseline
block_config                                                                            
combo_EFG               0.539398     0.525677  0.532538  0.009702              -0.021493
G_self_study            0.550363     0.531976  0.541169  0.013002              -0.012861
E_memo                  0.555005     0.530489  0.542747  0.017335              -0.011284
baseline                0.553842     0.554220  0.554031  0.000267               0.000000
F_grade_consistency     0.561340     0.548819  0.555080  0.008854               0.001049


INFO:22_ensemble_weight_tuning:
split                split_75_25  split_80_20      mean       std  mean_diff_vs_baseline
block_config                                                                            
combo_EFG               0.539398     0.525677  0.532538  0.009702              -0.021493
G_self_study            0.550363     0.531976  0.541169  0.013002              -0.012861
E_memo                  0.555005     0.530489  0.542747  0.017335              -0.011284
baseline                0.553842     0.554220  0.554031  0.000267               0.000000
F_grade_consistency     0.561340     0.548819  0.555080  0.008854               0.001049



■ アブレーション結果:
split                split_75_25  split_80_20      mean       std  mean_diff_vs_baseline
block_config                                                                            
combo_EFG               0.539398     0.525677  0.532538  0.009702              -0.021493
G_self_study            0.550363     0.531976  0.541169  0.013002              -0.012861
E_memo                  0.555005     0.530489  0.542747  0.017335              -0.011284
baseline                0.553842     0.554220  0.554031  0.000267               0.000000
F_grade_consistency     0.561340     0.548819  0.555080  0.008854               0.001049

※ このアブレーションはCPU同士の比較なので、GPU実測値(data/output/submit_result_report.md参照)とは直接比較しない


## 11. 総合結果・提出候補

split_80_20における各構成の提出ファイルパスを一覧化する。最も良い平均値を出した構成
（`combo_EFG`が単体最良ブロックを含みつつ悪化していなければ`combo_EFG`、そうでなければ
最良の単体ブロック構成）を提出候補とする。

In [18]:
split_80_20_rows = ablation_df[ablation_df["split"] == "split_80_20"].set_index("block_config")
summary_rows = split_80_20_rows[["val_score", "submission_path"]].reindex(
    ["baseline", "E_memo", "F_grade_consistency", "G_self_study", "combo_EFG"]
).reset_index()

logger.info("=" * 60)
logger.info("総合結果（split_80_20、提出候補一覧）")
logger.info("=" * 60)
logger.info("\n" + summary_rows.to_string())
print("\n■ 総合結果（split_80_20、提出候補一覧）:")
print(summary_rows.to_string(index=False))
print(f"\n(参考) 18_ CatBoost+D_expanded: Public 0.550352（現時点の最良）")

summary_rows

[2026-08-10 11:05:12] [INFO] ============================================================


INFO:22_ensemble_weight_tuning:============================================================


[2026-08-10 11:05:12] [INFO] 総合結果（split_80_20、提出候補一覧）


INFO:22_ensemble_weight_tuning:総合結果（split_80_20、提出候補一覧）


[2026-08-10 11:05:12] [INFO] ============================================================


INFO:22_ensemble_weight_tuning:============================================================


[2026-08-10 11:05:12] [INFO] 
          block_config  val_score                                                                                                                 submission_path
0             baseline   0.554220             /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_baseline.csv
1               E_memo   0.530489               /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_E_memo.csv
2  F_grade_consistency   0.548819  /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_F_grade_consistency.csv
3         G_self_study   0.531976         /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_G_self_study.csv
4            combo_EFG   0.525677            /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_combo_EFG.cs

INFO:22_ensemble_weight_tuning:
          block_config  val_score                                                                                                                 submission_path
0             baseline   0.554220             /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_baseline.csv
1               E_memo   0.530489               /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_E_memo.csv
2  F_grade_consistency   0.548819  /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_F_grade_consistency.csv
3         G_self_study   0.531976         /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_G_self_study.csv
4            combo_EFG   0.525677            /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_combo_EFG.


■ 総合結果（split_80_20、提出候補一覧）:
       block_config  val_score                                                                                                                submission_path
           baseline   0.554220            /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_baseline.csv
             E_memo   0.530489              /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_E_memo.csv
F_grade_consistency   0.548819 /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_F_grade_consistency.csv
       G_self_study   0.531976        /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_G_self_study.csv
          combo_EFG   0.525677           /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_split_80_20_combo_EFG.csv

(参考) 18_ CatBoost+D_ex

,block_config,val_score,submission_path
0,baseline,0.554220,/content/drive/MyDrive/jaggle_2026/data/output...
1,E_memo,0.530489,/content/drive/MyDrive/jaggle_2026/data/output...
2,F_grade_consistency,0.548819,/content/drive/MyDrive/jaggle_2026/data/output...
3,G_self_study,0.531976,/content/drive/MyDrive/jaggle_2026/data/output...
4,combo_EFG,0.525677,/content/drive/MyDrive/jaggle_2026/data/output...


## 12. アンサンブルの重みをパラメータチューニング

5つの構成（baseline / E_memo / F_grade_consistency / G_self_study / combo_EFG）の検証予測を使い、
Simple Average・ヒューリスティック重み（1/score）・`scipy.optimize`によるOptimized Weighted Average
（`12_`と同じ手法）の3種類を計算する。

**過去の教訓**: `12_`ではOptimized Weighted AverageがOOFで最良に見えたが、Public提出では単体モデルや
単純平均に劣った（過学習）。今回はこれを検知するため、**80/20・75/25の2つのsplitそれぞれで独立に
重みを最適化し、(a) 両splitで重みベクトル自体が似ているか、(b) 片方のsplitで最適化した重みをもう
片方のsplitの予測に適用した場合にスコアが大きく悪化しないか、の2点を確認する**。
どちらかが崩れていれば、Optimized Weighted Averageは過学習と判断しCPU/GPUの精度差を差し引いても
信用しない。combo_EFGが単体最良を明確に上回らない限り、単体モデルの提出を優先する。

In [19]:
ENSEMBLE_MODELS = ["baseline", "E_memo", "F_grade_consistency", "G_self_study", "combo_EFG"]

def load_val_preds_and_target(split_name, ratio):
    _, ag_tuning_data, _ = prepare_split(ratio, extra_blocks=set())
    y_va = ag_tuning_data[TARGET_COL].values
    preds = {}
    for block_name in ENSEMBLE_MODELS:
        row = ablation_df[(ablation_df["split"] == split_name) & (ablation_df["block_config"] == block_name)].iloc[0]
        preds[block_name] = np.load(row["val_pred_path"])
    return y_va, preds


def optimize_weights(y_va, preds_dict):
    preds_matrix = np.column_stack([preds_dict[m] for m in ENSEMBLE_MODELS])

    def neg_ll(w):
        w_norm = np.abs(w) / np.sum(np.abs(w))
        return log_loss(y_va, preds_matrix @ w_norm)

    result = minimize(neg_ll, x0=np.ones(len(ENSEMBLE_MODELS)) / len(ENSEMBLE_MODELS), method="Nelder-Mead")
    w_norm = np.abs(result.x) / np.sum(np.abs(result.x))
    return w_norm, neg_ll(result.x), preds_matrix


split_data = {}
for split_name, ratio in SPLIT_RATIOS.items():
    y_va, preds = load_val_preds_and_target(split_name, ratio)
    w_opt, opt_score, preds_matrix = optimize_weights(y_va, preds)

    simple_avg_score = log_loss(y_va, preds_matrix.mean(axis=1))
    scores = np.array([ablation_df[(ablation_df["split"] == split_name) & (ablation_df["block_config"] == m)]["val_score"].iloc[0] for m in ENSEMBLE_MODELS])
    heuristic_w = (1 / scores) / (1 / scores).sum()
    heuristic_score = log_loss(y_va, preds_matrix @ heuristic_w)

    single_best_model = ENSEMBLE_MODELS[np.argmin(scores)]
    single_best_score = scores.min()

    split_data[split_name] = {
        "y_va": y_va, "preds_matrix": preds_matrix, "w_opt": w_opt,
        "opt_score": opt_score, "simple_avg_score": simple_avg_score,
        "heuristic_score": heuristic_score, "single_best_model": single_best_model,
        "single_best_score": single_best_score,
    }

    logger.info(f"[{split_name}] single_best={single_best_model}({single_best_score:.6f}), "
                f"simple_avg={simple_avg_score:.6f}, heuristic_weighted={heuristic_score:.6f}, "
                f"optimized_weighted={opt_score:.6f}")
    print(f"\n■ [{split_name}]")
    print(f"  単体最良: {single_best_model} ({single_best_score:.6f})")
    print(f"  Simple Average: {simple_avg_score:.6f}")
    print(f"  Weighted Average(1/score): {heuristic_score:.6f}")
    print(f"  Optimized Weighted Average: {opt_score:.6f}")
    print(f"  Optimized Weights: " + ", ".join(f"{m}={w:.3f}" for m, w in zip(ENSEMBLE_MODELS, w_opt)))

[2026-08-10 11:05:13] [INFO] [split_80_20] single_best=combo_EFG(0.525677), simple_avg=0.525252, heuristic_weighted=0.524984, optimized_weighted=0.516915


INFO:22_ensemble_weight_tuning:[split_80_20] single_best=combo_EFG(0.525677), simple_avg=0.525252, heuristic_weighted=0.524984, optimized_weighted=0.516915



■ [split_80_20]
  単体最良: combo_EFG (0.525677)
  Simple Average: 0.525252
  Weighted Average(1/score): 0.524984
  Optimized Weighted Average: 0.516915
  Optimized Weights: baseline=0.000, E_memo=0.378, F_grade_consistency=0.000, G_self_study=0.398, combo_EFG=0.224
[2026-08-10 11:05:14] [INFO] [split_75_25] single_best=combo_EFG(0.539398), simple_avg=0.545589, heuristic_weighted=0.545470, optimized_weighted=0.537335


INFO:22_ensemble_weight_tuning:[split_75_25] single_best=combo_EFG(0.539398), simple_avg=0.545589, heuristic_weighted=0.545470, optimized_weighted=0.537335



■ [split_75_25]
  単体最良: combo_EFG (0.539398)
  Simple Average: 0.545589
  Weighted Average(1/score): 0.545470
  Optimized Weighted Average: 0.537335
  Optimized Weights: baseline=0.000, E_memo=0.000, F_grade_consistency=0.000, G_self_study=0.277, combo_EFG=0.723


In [20]:
print("■ 過学習チェック: 重みベクトルの一貫性")
w_8020 = split_data["split_80_20"]["w_opt"]
w_7525 = split_data["split_75_25"]["w_opt"]
weight_compare = pd.DataFrame({"split_80_20": w_8020, "split_75_25": w_7525}, index=ENSEMBLE_MODELS)
print(weight_compare)

print("\n■ 過学習チェック: 重みの相互適用（cross-apply）")
score_8020_with_8020_weights = split_data["split_80_20"]["opt_score"]
score_8020_with_7525_weights = log_loss(split_data["split_80_20"]["y_va"], split_data["split_80_20"]["preds_matrix"] @ w_7525)
score_7525_with_7525_weights = split_data["split_75_25"]["opt_score"]
score_7525_with_8020_weights = log_loss(split_data["split_75_25"]["y_va"], split_data["split_75_25"]["preds_matrix"] @ w_8020)

cross_apply_df = pd.DataFrame({
    "own_weights": [score_8020_with_8020_weights, score_7525_with_7525_weights],
    "other_split_weights": [score_8020_with_7525_weights, score_7525_with_8020_weights],
}, index=["split_80_20", "split_75_25"])
cross_apply_df["degradation"] = cross_apply_df["other_split_weights"] - cross_apply_df["own_weights"]
print(cross_apply_df)

logger.info("\n" + weight_compare.to_string())
logger.info("\n" + cross_apply_df.to_string())
print("\n※ degradationが大きい（ノイズ幅0.003〜0.006を超える）場合、重みは過学習しておりOptimized Weighted Averageは不採用とする")

■ 過学習チェック: 重みベクトルの一貫性
                      split_80_20   split_75_25
baseline             4.033299e-07  9.485552e-08
E_memo               3.775399e-01  1.940575e-04
F_grade_consistency  1.169130e-05  3.865566e-08
G_self_study         3.980468e-01  2.772449e-01
combo_EFG            2.244012e-01  7.225609e-01

■ 過学習チェック: 重みの相互適用（cross-apply）
             own_weights  other_split_weights  degradation
split_80_20     0.516915             0.520539     0.003623
split_75_25     0.537335             0.542627     0.005292
[2026-08-10 11:05:14] [INFO] 
                      split_80_20   split_75_25
baseline             4.033299e-07  9.485552e-08
E_memo               3.775399e-01  1.940575e-04
F_grade_consistency  1.169130e-05  3.865566e-08
G_self_study         3.980468e-01  2.772449e-01
combo_EFG            2.244012e-01  7.225609e-01


INFO:22_ensemble_weight_tuning:
                      split_80_20   split_75_25
baseline             4.033299e-07  9.485552e-08
E_memo               3.775399e-01  1.940575e-04
F_grade_consistency  1.169130e-05  3.865566e-08
G_self_study         3.980468e-01  2.772449e-01
combo_EFG            2.244012e-01  7.225609e-01


[2026-08-10 11:05:14] [INFO] 
             own_weights  other_split_weights  degradation
split_80_20     0.516915             0.520539     0.003623
split_75_25     0.537335             0.542627     0.005292


INFO:22_ensemble_weight_tuning:
             own_weights  other_split_weights  degradation
split_80_20     0.516915             0.520539     0.003623
split_75_25     0.537335             0.542627     0.005292



※ degradationが大きい（ノイズ幅0.003〜0.006を超える）場合、重みは過学習しておりOptimized Weighted Averageは不採用とする


### 提出候補の決定

`combo_EFG`（またはE_memo/G_self_study）を単体で提出する方針は変えず、アンサンブルは
**過学習チェック（重みの一貫性・cross-apply degradation）を両方クリアした場合のみ**追加の
提出候補として検討する。クリアしない場合は、これまで通り単体モデルの提出を優先する。

In [21]:
test_preds_split_80_20 = {}
for block_name in ENSEMBLE_MODELS:
    row = split_80_20_rows.loc[block_name]
    test_preds_split_80_20[block_name] = pd.read_csv(
        row["submission_path"], header=None, names=[ID_COL, TARGET_COL]
    ).set_index(ID_COL)[TARGET_COL]

w_opt_80_20 = split_data["split_80_20"]["w_opt"]
blend = sum(test_preds_split_80_20[m] * w for m, w in zip(ENSEMBLE_MODELS, w_opt_80_20))
ensemble_sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_optimized_weighted_split_80_20.csv"
pd.DataFrame({ID_COL: blend.index, TARGET_COL: blend.values}).to_csv(ensemble_sub_path, index=False, header=False)
logger.info(f"Optimized Weighted Average(split_80_20)提出ファイル保存: {ensemble_sub_path}")
print(f"提出ファイル: {ensemble_sub_path}")

[2026-08-10 11:05:15] [INFO] Optimized Weighted Average(split_80_20)提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_optimized_weighted_split_80_20.csv


INFO:22_ensemble_weight_tuning:Optimized Weighted Average(split_80_20)提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_optimized_weighted_split_80_20.csv


提出ファイル: /content/drive/MyDrive/jaggle_2026/data/output/20260810/20260810_22_ensemble_weight_tuning_optimized_weighted_split_80_20.csv


## 13. まとめ・次のアクション

1. アブレーション結果の表（10節、CPU同士）で、E/F/G単体それぞれがbaselineに対し明確な改善を
   示したか確認する（GPU実測値とは直接比較しない）。
2. アンサンブルの過学習チェック（12節）: 重みベクトルが両splitで似ているか、cross-apply degradationが
   小さいかを確認する。どちらかが崩れていればOptimized Weighted Averageは不採用とする。
3. 提出優先度は **E_memo単体 > G_self_study単体**。過学習チェックをクリアしたアンサンブルがあれば、
   それも追加の提出候補とする。
4. 結果が出たら`data/output/submit_result_report.md`に追記し、メモリ（`best_submission_status.md`・
   `ensemble_oof_overfitting.md`）も更新する。

### バックログ（今回は着手しない）
- テキスト特徴量を日本語の事前学習済み文埋め込みモデルに置き換える案（`19_`で一度試して不採用、将来再検討の余地）